In [2]:
import numpy as np
import pandas as pd 

In [3]:
df = pd.read_csv("../data/Divar.csv")

print(df.head(5))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_13212\69017167.py:1: DtypeWarning: Columns (0: rent_to_single, 1: floor, 2: total_floors_count, 3: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Divar.csv")


   Unnamed: 0         cat2_slug       cat3_slug city_slug neighborhood_slug  \
0           0    temporary-rent           villa     karaj         mehrshahr   
1           1  residential-sell  apartment-sell    tehran           gholhak   
2           2  residential-rent  apartment-rent    tehran             tohid   
3           3   commercial-rent     office-rent    tehran          elahiyeh   
4           4  residential-sell  apartment-sell   mashhad          emamreza   

      created_at_month    user_type  \
0  2024-08-01 00:00:00  مشاور املاک   
1  2024-05-01 00:00:00  مشاور املاک   
2  2024-10-01 00:00:00          NaN   
3  2024-06-01 00:00:00          NaN   
4  2024-05-01 00:00:00  مشاور املاک   

                                         description  \
0  ۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...   
1  دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...   
2                                    تخلیه پایان ماه   
3  فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...   
4  هلدینگ 

<div dir="rtl">

### بررسی و پاک‌سازی ستون‌های مرتبط با قابلیت تبدیل رهن و اجاره

در این مرحله چهار ستون زیر بررسی شدند:

`rent_credit_transform` ، `transformable_price` ، `transformable_credit` و `transformable_rent`

- ستون‌های `rent_credit_transform` و `transformable_price` در ابتدا از نوع `object` بودند.
- بررسی نوع واقعی مقادیر نشان داد که مقدارهای غیرخالی آن‌ها از نوع بولین واقعی (`bool`) هستند.
- نوع این دو ستون به `boolean` قابل‌خالی‌بودن تبدیل شد تا سه حالت `True`، `False` و `<NA>` حفظ شود.
- تعداد مقادیر `True`، `False` و خالی قبل و بعد از تبدیل بدون تغییر باقی ماند.

- ستون‌های `transformable_credit` و `transformable_rent` از نوع `float64` هستند و مقادیر مالی نگه می‌دارند.
- `transformable_credit` تقریباً همیشه با `credit_value` برابر است.
- `transformable_rent` نیز تقریباً همیشه با `rent_value` برابر است و فقط در تعداد بسیار کمی ردیف اختلاف مشاهده شد.
- خالی بودن ستون‌های `transformable_credit` و `transformable_rent` عمدتاً با خالی بودن `rent_credit_transform` مرتبط است؛ بنابراین این مقادیر خالی ساختاری در نظر گرفته شدند.
- در نتیجه، برای این دو ستون عملیات `fillna` انجام نشد و مقادیرشان بدون تغییر باقی ماند.

### نتیجه

نوع داده‌ی دو ستون منطقی اصلاح شد و دو ستون مالی نیز پس از بررسی، بدون دستکاری مقدارها نگه داشته شدند.

</div>

In [4]:
cols = ["rent_credit_transform", "transformable_price"]

for col in cols:
    print("=" * 60)
    print("Column:", col)

    print("\nPython value types:")
    print(
        df[col]
        .dropna()
        .map(type)
        .value_counts()
    )

    print("\nValues with Python types:")
    print(
        df.loc[df[col].notna(), col]
        .drop_duplicates()
        .map(lambda x: (repr(x), type(x).__name__))
    )


Column: rent_credit_transform

Python value types:
rent_credit_transform
<class 'bool'>    352985
Name: count, dtype: int64

Values with Python types:
2     (False, bool)
11     (True, bool)
Name: rent_credit_transform, dtype: object
Column: transformable_price

Python value types:
transformable_price
<class 'bool'>    352894
Name: count, dtype: int64

Values with Python types:
2    (False, bool)
5     (True, bool)
Name: transformable_price, dtype: object


In [5]:
cols = ["rent_credit_transform", "transformable_price"]

allowed_values = {
    True,
    False,
    "True",
    "False"
}

for col in cols:
    unexpected = df.loc[
        df[col].notna() & ~df[col].isin(allowed_values),
        col
    ].unique()

    if len(unexpected) > 0:
        raise ValueError(
            f"Unexpected values in {col}: {unexpected}"
        )

    df[col] = df[col].replace({
        "True": True,
        "False": False
    })

    df[col] = df[col].astype("boolean")

In [6]:
for col in cols:
    print("=" * 60)
    print("Column:", col)
    print("Data type:", df[col].dtype)
    print(df[col].value_counts(dropna=False))

Column: rent_credit_transform
Data type: boolean
rent_credit_transform
<NA>     647015
False    326561
True      26424
Name: count, dtype: Int64
Column: transformable_price
Data type: boolean
transformable_price
<NA>     647106
False    279778
True      73116
Name: count, dtype: Int64


مرحله دوم: آنالیر دیتاهای ستون‌های 

transformable_credit
transformed_credit
transformable_rent
transformed_rent


In [7]:
summary = pd.DataFrame({
    col: {
        "missing": df[col].isna().sum(),
        "zero": (df[col] == 0).sum(),
        "negative": (df[col] < 0).sum(),
        "positive": (df[col] > 0).sum()
    }
    for col in cols
}).T

print(summary)

                       missing    zero  negative  positive
rent_credit_transform   647015  326561         0     26424
transformable_price     647106  279778         0     73116


<div dir="rtl">

چهار ستون مبلغ‌های مرتبط با تبدیل رهن و اجاره فعلاً بدون تغییر حفظ شدند. 
زیادبودن مقادیر خالی، به‌تنهایی دلیل حذف یا پرکردن آن‌ها نیست و ممکن است ناشی از کاربرد محدود این اطلاعات باشد. بررسی تکمیلی و تصمیم دربارهٔ استفاده از آن‌ها، با توجه به نیاز مدل و احتمال نشت اطلاعات، به مراحل بعد موکول شد.


</div>










در این بخش قرار است یک سری دیتا های فروش ، اجاره ، رهن را بررسی کنیم 


In [8]:
rental_rows = df[
    df["cat2_slug"].str.contains("rent", case=False, na=False)
].copy()

print("Total rental rows:", len(rental_rows))

print("\nprice_value:")
print("Missing:", rental_rows["price_value"].isna().sum())
print("Zero:", (rental_rows["price_value"] == 0).sum())
print("Negative:", (rental_rows["price_value"] < 0).sum())
print("Positive:", (rental_rows["price_value"] > 0).sum())

print("\nrent_type:")
print(rental_rows["rent_type"].value_counts(dropna=False))

print("\ncredit_value:")
print("Missing:", rental_rows["credit_value"].isna().sum())
print("Zero:", (rental_rows["credit_value"] == 0).sum())
print("Negative:", (rental_rows["credit_value"] < 0).sum())
print("Positive:", (rental_rows["credit_value"] > 0).sum())

print("\nrent_value:")
print("Missing:", rental_rows["rent_value"].isna().sum())
print("Zero:", (rental_rows["rent_value"] == 0).sum())
print("Negative:", (rental_rows["rent_value"] < 0).sum())
print("Positive:", (rental_rows["rent_value"] > 0).sum())

Total rental rows: 383028

price_value:
Missing: 382983
Zero: 0
Negative: 0
Positive: 45

rent_type:
rent_type
NaN            279074
rent_credit    103807
full_credit       147
Name: count, dtype: int64

credit_value:
Missing: 30941
Zero: 2864
Negative: 0
Positive: 349223

rent_value:
Missing: 31714
Zero: 59240
Negative: 0
Positive: 292074


In [9]:
zero_rent = rental_rows[rental_rows["rent_value"] == 0].copy()

print("Total zero-rent rows:", len(zero_rent))

print("\nrent_type:")
print(zero_rent["rent_type"].value_counts(dropna=False))

print("\nrent_mode:")
print(zero_rent["rent_mode"].value_counts(dropna=False))

print("\ncredit_mode:")
print(zero_rent["credit_mode"].value_counts(dropna=False))

print("\ncredit_value summary:")
print("Missing:", zero_rent["credit_value"].isna().sum())
print("Zero:", (zero_rent["credit_value"] == 0).sum())
print("Negative:", (zero_rent["credit_value"] < 0).sum())
print("Positive:", (zero_rent["credit_value"] > 0).sum())

print("\nrent_credit_transform:")
print(zero_rent["rent_credit_transform"].value_counts(dropna=False))

Total zero-rent rows: 59240

rent_type:
rent_type
NaN            37453
rent_credit    21640
full_credit      147
Name: count, dtype: int64

rent_mode:
rent_mode
مجانی    59240
Name: count, dtype: int64

credit_mode:
credit_mode
مقطوع     59222
توافقی       18
Name: count, dtype: int64

credit_value summary:
Missing: 18
Zero: 0
Negative: 0
Positive: 59222

rent_credit_transform:
rent_credit_transform
False    52572
True      6668
Name: count, dtype: Int64


In [10]:
print("Total zero-rent rows:", len(zero_rent))

print("\nprice_mode:")
print(zero_rent["price_mode"].value_counts(dropna=False))

print("\nprice_value:")
print("Missing:", zero_rent["price_value"].isna().sum())
print("Zero:", (zero_rent["price_value"] == 0).sum())
print("Negative:", (zero_rent["price_value"] < 0).sum())
print("Positive:", (zero_rent["price_value"] > 0).sum())

print(
    "\nRows with any sale-price information:",
    (
        zero_rent["price_mode"].notna()
        | zero_rent["price_value"].notna()
    ).sum()
)

Total zero-rent rows: 59240

price_mode:
price_mode
NaN    59240
Name: count, dtype: int64

price_value:
Missing: 59240
Zero: 0
Negative: 0
Positive: 0

Rows with any sale-price information: 0


<div dir="rtl">

### خلاصه بررسی رهن کامل

- کل دیتاست: **1,000,000 آگهی**
- فقط **147 آگهی** به‌صورت مستقیم `full_credit` ثبت شده بودند.
- **59,240 آگهی** دارای `rent_value = 0` بودند.
- از این تعداد، **59,222 آگهی** مبلغ رهن مثبت داشتند.
- هر **59,240 آگهی** هیچ اطلاعاتی از قیمت فروش نداشتند.
- در نتیجه مشخص شد `rent_type` به‌تنهایی برای تشخیص رهن کامل قابل اتکا نیست.
- یک ستون جدید به نام `inferred_full_credit` ساخته شد.
- نتیجه نهایی:
  - `True`: **59,234**
  - `False`: **865,687**
  - `<NA>`: **75,079**
- ستون اصلی `rent_type` بدون تغییر نگه داشته شد.

</div>

In [11]:
df["inferred_full_credit"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="boolean"
)

rental_category = df["cat2_slug"].str.contains(
    "rent",
    case=False,
    na=False
)

explicit_full_credit = (
    df["rent_type"] == "full_credit"
)

strong_full_credit = (
    rental_category
    & (df["rent_value"] == 0)
    & (df["credit_value"] > 0)
    & df["price_mode"].isna()
    & df["price_value"].isna()
)

df.loc[
    explicit_full_credit | strong_full_credit,
    "inferred_full_credit"
] = True

definitely_not_full_credit = (
    (df["rent_value"] > 0)
    | df["price_value"].notna()
    | df["price_mode"].notna()
)

df.loc[
    definitely_not_full_credit & ~explicit_full_credit,
    "inferred_full_credit"
] = False

In [12]:
definitely_not_full_credit = (
    (df["rent_value"] > 0)
    | df["price_value"].notna()
    | df["price_mode"].notna()
)

df.loc[
    definitely_not_full_credit & ~explicit_full_credit,
    "inferred_full_credit"
] = False

In [13]:
print(df["inferred_full_credit"].value_counts(dropna=False))

inferred_full_credit
False    865687
<NA>      75079
True      59234
Name: count, dtype: Int64


In [14]:
print(
    pd.crosstab(
        df["rent_type"].fillna("missing"),
        df["inferred_full_credit"].astype("string").fillna("missing")
    )
)

inferred_full_credit   False   True  missing
rent_type                                   
full_credit                0    147        0
missing               785131  37453    73455
rent_credit            80556  21634     1624


<div dir="rtl">

### پاک‌سازی مقادیر پرت ستون اجاره

برای جلوگیری از تأثیر شدید مقادیر غیرعادی بر تحلیل‌های آماری، مقادیر پرت ستون `rent_value` بررسی شدند.

آستانه پرت بر اساس صدک ۹۹.۵ داده‌ها تعیین شد که برابر با **400,000,000** بود.

در نتیجه:

- تعداد مقادیر پرت شناسایی‌شده: **1,712**
- بیشترین مقدار اولیه `rent_value`: **999,999,999,999,999**
- بیشترین مقدار در نسخه تمیزشده: **400,000,000**
- مقادیر پرت از ستون اصلی حذف نشدند و ستون `rent_value` بدون تغییر نگه داشته شد.
- یک ستون جدید با نام `rent_value_clean` ساخته شد که در آن مقادیر پرت به `NaN` تبدیل شدند.
- تعداد مقادیر گم‌شده از **648,678** به **650,390** رسید که دقیقاً معادل اضافه شدن همان **1,712** مقدار پرت است.

در نتیجه، ستون `rent_value_clean` به‌عنوان نسخه مناسب‌تر برای تحلیل‌هایی مانند محاسبه میانگین اجاره استفاده می‌شود، در حالی که داده خام اصلی همچنان حفظ شده است.

</div>

In [15]:
# Flag extreme rent values without modifying the original column

rent_upper_limit = df["rent_value"].quantile(0.995)

df["rent_value_outlier"] = (
    df["rent_value"].notna()
    & (df["rent_value"] > rent_upper_limit)
)

print("Rent outlier threshold:", rent_upper_limit)
print(df["rent_value_outlier"].value_counts(dropna=False))

Rent outlier threshold: 400000000.0
rent_value_outlier
False    998288
True       1712
Name: count, dtype: int64


In [16]:
print("Outlier rows:", df["rent_value_outlier"].sum())

Outlier rows: 1712


In [17]:
# Detect extreme rent values
rent_upper_limit = df["rent_value"].quantile(0.995)

df["rent_value_outlier"] = (
    df["rent_value"].notna()
    & (df["rent_value"] > rent_upper_limit)
)

# Create cleaned rent column
df["rent_value_clean"] = df["rent_value"].mask(
    df["rent_value_outlier"]
)

print("Outlier threshold:", rent_upper_limit)
print("Outliers detected:", df["rent_value_outlier"].sum())
print("Original non-null:", df["rent_value"].notna().sum())
print("Clean non-null:", df["rent_value_clean"].notna().sum())

Outlier threshold: 400000000.0
Outliers detected: 1712
Original non-null: 351322
Clean non-null: 349610
